In [12]:
# @title Package installation
from scipy import sparse                     # For sparse.csr_matrix
import pandas as pd                          # For dataframes
import numpy as np                           # For the math
import requests                              # To fetch the CSVs from GitHub
import gc                                    # For our RAM-saving strategy


from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize


from sklearn.feature_extraction.text import TfidfVectorizer # For the first content similarity model
from sentence_transformers import SentenceTransformer # Final content similarity model

In [13]:
# @title Importing the data

# Set username and repository name
user = "raniabenhamidane-beep"
repo = "ML-libraryGE"
folder_path = "data"
api_url = f"https://api.github.com/repos/{user}/{repo}/contents/{folder_path}"
response = requests.get(api_url)

# We then check if the request was successful
if response.status_code == 200:
    file_list = response.json()
    dataframes = {}

    for file in file_list:
        # Here we only keep the CSV files. We don't need the README.md for our coding.
        if file['name'].endswith('.csv'):
            file_name = file['name']
            raw_url = file['download_url']

            # Explicitly handle the Mac-exported file with a semicolon delimiter
            if file_name == 'items_enriched.csv':
                print(f"Loading {file_name} with Mac delimiter (;)...")
                dataframes[file_name] = pd.read_csv(raw_url, sep=';')
            else:
                # For all other files, try the default comma separator first
                try:
                    dataframes[file_name] = pd.read_csv(raw_url, sep=',')
                except pd.errors.ParserError:
                    # Keep a fallback just in case other files also use semicolons
                    dataframes[file_name] = pd.read_csv(raw_url, sep=';')

    print("\nSuccessfully loaded:")
    for name in dataframes.keys():
        print(f"- {name}")
else:
    print(f"Failed to fetch data. Status code: {response.status_code}")

# Loading the datasets.
interactions = dataframes.get('interactions_train.csv')
items = dataframes.get('items_enriched_api.csv') # We use the enriched items.csv for better predictions, although the result is fairly similar.


Successfully loaded:
- interactions_train.csv
- item_prediction_hybrid.csv
- items_enriched_api.csv
- itemsclean_with_metadata.csv


In [14]:
# @title Creation of the data matrix using interactions: global temporal split and user time decay

# Global temporal split (ie. ranks borrowed books from oldest to newest, completely ignoring who made the borrow.)
interactions = interactions.sort_values("t")
interactions["pct_rank"] = interactions["t"].rank(pct=True, method='dense')
train_data = interactions.copy()
train_users = set(train_data['u'].unique())

# Now we go to to creating the data matrix
n_users = interactions.u.nunique()
n_items = items.i.nunique()

# We have time decay within our data matrix
# 1. Copy the dataframe
train_data_decay = train_data.copy()

# 2. Map each user's timeline on a scale from 0.0 (oldest) to 1.0 (newest)
train_data_decay['user_time_rank'] = train_data_decay.groupby('u')['t'].rank(pct=True, method='dense')

# 3. Apply the decay formula : decay the weights down to a minimum of 0.2 (based onsensitivity analysis with train / test split
# this decay performs best)
# 0.5 is a gentle decay, 0.1 is aggressive!)
min_weight = 0.2
train_data_decay['decay_weight'] = min_weight + ((1.0 - min_weight) * train_data_decay['user_time_rank'])

# 4. Redefine our matrix builder to use these new weights instead of binary 1s
def create_weighted_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    # Instead of placing a '1', we place the exact calculated decay weight!
    data_matrix[data["u"].values, data["i"].values] = data["decay_weight"].values
    return data_matrix

print("Building the time-weighted interaction matrix...")
train_data_matrix_decay = create_weighted_data_matrix(train_data_decay, n_users, n_items)
train_data_matrix_decay = sparse.csr_matrix(train_data_matrix_decay)

Building the time-weighted interaction matrix...


**A comment on the global temporal split vs user time decay**


The two steps actually work together: they operate at two completely different stages of our pipeline.
The global split defines the world our model is allowed to see, while the time decay defines how much the model should listen to specific events within that world.

# First content similarity model

In [20]:
# 1. Sort the items so row 0 is exactly item 0, ensuring it aligns perfectly with our train_matrix
items_sorted = items.sort_values('i').reset_index(drop=True)

# 2. Fill any missing values with an empty string so the math doesn't crash
titles = items_sorted['Title'].fillna("")
authors = items_sorted['Author'].fillna("")
subjects = items_sorted['Subjects'].fillna("")

# 3. Combine Title, Author, and Subjects into one "mega-string" for the NLP engine
text_data = titles + " " + authors + " " + subjects

# 4. Convert the text to math
# Limit it to 5000 features to keep Colab RAM perfectly safe
tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(text_data)

# 5. Calculate similarity based strictly on the content!
content_similarity = cosine_similarity(tfidf_matrix)

# The original hybrid 3 way model

In [21]:
def ultimate_3way_ensemble(train_matrix, item_sim_matrix, content_sim_matrix, filename,
                           k=50, batch_size=250,
                           u2u_weight=0.4, i2i_weight=0.4, content_weight=0.2):

    n_users = train_matrix.shape[0]
    all_recommendations = []

    print("Calculating global item popularity for cold-start users...")
    item_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    top_global_items = np.argsort(item_popularity)[-10:][::-1]
    top_global_string = " ".join(map(str, top_global_items))

    # --- UPDATED: Pre-calculate denominators safely for sparse matrices ---
    if hasattr(item_sim_matrix, 'toarray'):
        item_sim_sum = np.array(np.abs(item_sim_matrix).sum(axis=1)).flatten() + 1e-9
    else:
        item_sim_sum = np.array(np.abs(item_sim_matrix).sum(axis=1)).flatten() + 1e-9

    content_sim_sum = np.array(np.abs(content_sim_matrix).sum(axis=1)).flatten() + 1e-9

    print(f"Starting 3-way ensemble predictions for {n_users} users...")

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)
        train_batch = train_matrix[start_idx:end_idx]

        # ==========================================
        # 1. USER-TO-USER
        # ==========================================
        u2u_sim_batch = cosine_similarity(train_batch, train_matrix)
        for i in range(u2u_sim_batch.shape[0]):
            row = u2u_sim_batch[i]
            if len(row) > k + 1:
                threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
                threshold_val = row[threshold_idx]
                row[row < threshold_val] = 0

        u2u_sim_sparse = sparse.csr_matrix(u2u_sim_batch)
        u2u_weighted_sum = u2u_sim_sparse.dot(train_matrix)
        u2u_sum_sim = np.array(u2u_sim_sparse.sum(axis=1)).flatten() + 1e-9

        if hasattr(u2u_weighted_sum, 'toarray'):
            u2u_pred = u2u_weighted_sum.toarray() / u2u_sum_sim[:, None]
        else:
            u2u_pred = u2u_weighted_sum / u2u_sum_sim[:, None]

        # ==========================================
        # 2. ITEM-TO-ITEM
        # ==========================================
        i2i_weighted_sum = train_batch.dot(item_sim_matrix)

        if hasattr(i2i_weighted_sum, 'toarray'):
            i2i_pred = i2i_weighted_sum.toarray() / item_sim_sum
        else:
            i2i_pred = i2i_weighted_sum / item_sim_sum

        # ==========================================
        # 3. CONTENT-BASED
        # ==========================================
        content_weighted_sum = train_batch.dot(content_sim_matrix)

        if hasattr(content_weighted_sum, 'toarray'):
            content_pred = content_weighted_sum.toarray() / content_sim_sum
        else:
            content_pred = content_weighted_sum / content_sim_sum

        # ==========================================
        # 4. THE BLEND
        # ==========================================
        ensemble_pred = (u2u_pred * u2u_weight) + (i2i_pred * i2i_weight) + (content_pred * content_weight)

        # ==========================================
        # 5. TOP 10 & POPULARITY FALLBACK
        # ==========================================
        top_10_indices = np.argsort(ensemble_pred, axis=1)[:, -10:][:, ::-1]

        for i in range(top_10_indices.shape[0]):
            if np.max(ensemble_pred[i]) == 0:
                all_recommendations.append(top_global_string)
            else:
                recs = " ".join(map(str, top_10_indices[i]))
                all_recommendations.append(recs)

        # ==========================================
        # 6. RAM CLEANUP
        # ==========================================
        del u2u_sim_batch, u2u_sim_sparse, u2u_weighted_sum, u2u_pred
        del i2i_weighted_sum, i2i_pred
        del content_weighted_sum, content_pred
        del ensemble_pred, top_10_indices
        gc.collect()

    print("\nBuilding submission file...")
    df_submission = pd.DataFrame({
        'user_id': range(len(all_recommendations)),
        'recommendation': all_recommendations
    })

    df_submission.to_csv(filename, index=False)
    print(f"File saved successfully: {filename}")

    return df_submission

def apply_knn_inplace(sim_matrix, k=50):
    print(f"Applying KNN filter (k={k}) to eliminate noise...")
    for i in range(sim_matrix.shape[0]):
        row = sim_matrix[i]
        if len(row) > k + 1:
            # Find the threshold value for the top k elements
            threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
            threshold_val = row[threshold_idx]
            # Zero out anything below the threshold
            row[row < threshold_val] = 0
    return sim_matrix


# --- EXECUTE THE CODE ---
print("Computing global item similarity...")
item_similarity = cosine_similarity(train_data_matrix_decay.T)

# 1. Filter out the noise! Only keep the top 50 closest items
item_similarity = apply_knn_inplace(item_similarity, k=50)

# 2. CRITICAL MEMORY FIX: Compress the newly zeroed-out matrix!
item_similarity_sparse = sparse.csr_matrix(item_similarity)

# 3. The basic content similarity prediction
del item_similarity
gc.collect()

# 3. Run the 3-way ensemble with the clean, compressed item matrix
ensemble_submission = ultimate_3way_ensemble(
    train_matrix=train_data_matrix_decay,
    item_sim_matrix=item_similarity_sparse, # Pass the sparse version!
    content_sim_matrix=content_similarity,
    filename='3way_ensemble_item_knn.csv',
    k=50,
    batch_size=250,
    u2u_weight=0.4,
    i2i_weight=0.4,
    content_weight=0.2
)

try:
    from google.colab import files
    files.download('3way_ensemble_item_knn.csv')
except:
    pass

Computing global item similarity...
Applying KNN filter (k=50) to eliminate noise...
Calculating global item popularity for cold-start users...
Starting 3-way ensemble predictions for 7838 users...

Building submission file...
File saved successfully: 3way_ensemble_item_knn.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Key points from our first attempts

Playing around with **user and item similarity models** (from the Recommenders lab session), as well as a **basic content similarity model**, we decided to implement a **3 way hybrid model** to get the best out of each model and ultimately improve our accuracy.

Another finding was to make estimates of our accuracy using a train-test split on our working file, however to **make submissions on the kaggle leaderboard predicted using the full interactions dataframe**. This may seem obvious, but we weren't sure we were allowed to do so!

This led us to our first **good result of 0.1644**. Adding the **knn model** to the user similarity helped this value improve a little bit to an **accuracy of 0.1655**.

## The key takeaways:
* The **global popularity fallback performs best**.
* The sparse matrix helps to **deal with the RAM build-up**, as well as an agressive cleanup within loops once the similarity has been computed.
* The **knn model added to the user similarity model improves/maintains the accuracy** of the hybrid model, although it doesn't improve the  

## Next steps of the hybrid model to go beyond the 0.17 accuracy  floor.
* Embedding the content using a multilingual Sentence Transformers (because most books are in french)
* Trying the sparse dot product instead of cosine similarity
* Finetuning the value of k, the weighting between the models and the minimum weight of the time decay matrix



# Upgrades from the baseline hybrid

In [22]:
# @title Trying another content similarity model with a multilingual model: 'mpnet'

# 1. Swapped to the smarter 'mpnet' model for higher accuracy
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# 2. Prepare the dataframe items
items_sorted = items.sort_values('i').reset_index(drop=True)
titles = items_sorted['Title'].fillna("")
authors = items_sorted['Author'].fillna("")
subjects = items_sorted['Subjects'].fillna("")

# 2. We format it like a mini-sentence so the AI understands it perfectly
text_data = "Titre: " + titles + ". Auteur: " + authors + ". Sujets: " + subjects + "."

# 3. From text to math !
embeddings = model.encode(text_data.tolist(), show_progress_bar=True)

# 3. optional. --- To save embeddings, add them to Drive and not have to wait every time---
# print("Saving embeddings to disk...")
# np.save('my_smart_embeddings.npy', embeddings)
# drive.mount('/content/drive')
# embeddings = np.load('/content/drive/MyDrive/my_smart_embeddings.npy')
# ------------------------------------

# 4. Computing similarity semantics
content_similarity_dense = cosine_similarity(embeddings)

# 5. We use the knn helper function to filter out the static noise!
content_similarity_filtered = apply_knn_inplace(content_similarity_dense, k=50)

# 6. Compression of RAM
content_similarity_sparse = sparse.csr_matrix(content_similarity_filtered)

del content_similarity_dense
del content_similarity_filtered

gc.collect()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/478 [00:00<?, ?it/s]

Applying KNN filter (k=50) to eliminate noise...


16

## Good news for the new content similarity model:
The SentenceTransformers performs better!!

## Next steps (done synchronously):

**Step 1** We want to **optimise the speed of computing** similarities to be able to try different weightings.

**Step 2** We  **replace the cosine similarity with the .dot product**: it performs better!
Why? While cosine similarity normalizes for the number of interactions, the raw dot product does not. In recommendation systems, this often helps because it naturally introduces a popularity bias. Users who have many interactions are treated as more "informative" neighbors. It suggests our dataset benefits from weighting active users and popular items more heavily. To lean into this success, we should shift more weight toward the User-to-User (U2U) component.

## The following model:
U2U is computed using sparse dot product within the 3 way, input of I2I and content similarities will also be computed this way (but outside the model).  

In [23]:
# @title Time-efficient & sparse dot product hybrid model
def hyper_fast_weight_search(train_matrix, item_sim_matrix, content_sim_matrix, weight_combos, k=50, batch_size=250):
    n_users = train_matrix.shape[0]

    # Dictionary to hold the lists of recommendations for EACH weight combo
    all_recs = {weights: [] for weights in weight_combos}

    print("Calculating global item popularity for cold-start users...")
    item_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    top_global_items = np.argsort(item_popularity)[-10:][::-1]
    top_global_string = " ".join(map(str, top_global_items))

    # Pre-calculate denominators safely for sparse matrices
    if hasattr(item_sim_matrix, 'toarray'):
        item_sim_sum = np.array(np.abs(item_sim_matrix).sum(axis=1)).flatten() + 1e-9
    else:
        item_sim_sum = np.array(np.abs(item_sim_matrix).sum(axis=1)).flatten() + 1e-9

    content_sim_sum = np.array(np.abs(content_sim_matrix).sum(axis=1)).flatten() + 1e-9

    print(f"Starting hyper-fast grid search for {len(weight_combos)} combinations...")

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)
        train_batch = train_matrix[start_idx:end_idx]

        # ==========================================
        # 1. THE HEAVY MATH (Calculated 1x per batch to save time)
        # ==========================================

        # U2U: Replaced cosine_similarity with pure sparse Dot Product
        # .T transposes the matrix to align user vectors.
        # .toarray() ensures your k-NN sorting loop below still works.
        u2u_sim_batch = train_batch.dot(train_matrix.T)
        if hasattr(u2u_sim_batch, 'toarray'):
            u2u_sim_batch = u2u_sim_batch.toarray()

        for i in range(u2u_sim_batch.shape[0]):
            row = u2u_sim_batch[i]
            if len(row) > k + 1:
                threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
                threshold_val = row[threshold_idx]
                row[row < threshold_val] = 0

        u2u_sim_sparse = sparse.csr_matrix(u2u_sim_batch)
        u2u_weighted_sum = u2u_sim_sparse.dot(train_matrix)
        u2u_sum_sim = np.array(u2u_sim_sparse.sum(axis=1)).flatten() + 1e-9

        if hasattr(u2u_weighted_sum, 'toarray'):
            u2u_pred = u2u_weighted_sum.toarray() / u2u_sum_sim[:, None]
        else:
            u2u_pred = u2u_weighted_sum / u2u_sum_sim[:, None]

        # I2I
        i2i_weighted_sum = train_batch.dot(item_sim_matrix)
        if hasattr(i2i_weighted_sum, 'toarray'):
            i2i_pred = i2i_weighted_sum.toarray() / item_sim_sum
        else:
            i2i_pred = i2i_weighted_sum / item_sim_sum

        # CONTENT (AI Embeddings)
        content_weighted_sum = train_batch.dot(content_sim_matrix)
        if hasattr(content_weighted_sum, 'toarray'):
            content_pred = content_weighted_sum.toarray() / content_sim_sum
        else:
            content_pred = content_weighted_sum / content_sim_sum

        # ==========================================
        # 2. THE FAST WEIGHT LOOP
        # ==========================================
        for (w_u, w_i, w_c) in weight_combos:

            ensemble_pred = (u2u_pred * w_u) + (i2i_pred * w_i) + (content_pred * w_c)
            top_10_indices = np.argsort(ensemble_pred, axis=1)[:, -10:][:, ::-1]

            for i in range(top_10_indices.shape[0]):
                if np.max(ensemble_pred[i]) == 0:
                    all_recs[(w_u, w_i, w_c)].append(top_global_string)
                else:
                    recs = " ".join(map(str, top_10_indices[i]))
                    all_recs[(w_u, w_i, w_c)].append(recs)

        # ==========================================
        # 3. RAM CLEANUP
        # ==========================================
        del u2u_sim_batch, u2u_sim_sparse, u2u_weighted_sum, u2u_pred
        del i2i_weighted_sum, i2i_pred, content_weighted_sum, content_pred
        try: del ensemble_pred, top_10_indices
        except: pass
        gc.collect()

    print("\nBuilding submission files...")
    for weights, recs in all_recs.items():
        filename = f"ai_ensemble_{weights[0]}_{weights[1]}_{weights[2]}.csv"
        df_submission = pd.DataFrame({
            'user_id': range(len(recs)),
            'recommendation': recs
        })
        df_submission.to_csv(filename, index=False)
        print(f"File saved successfully: {filename}")

        # Try to trigger the download!
        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

In [25]:
# @title Item similarity with sparse dot product

def fast_item_similarity(train_matrix, top_k=100):
    print("Normalizing item vectors...")
    # Transpose so rows are items, columns are users.
    # normalize() makes the length of each item's vector exactly 1.
    item_vectors_norm = normalize(train_matrix.T, norm='l2', axis=1)

    print("Computing sparse dot product (this is identical to cosine similarity)...")
    # Multiplying a sparse matrix by its transpose is heavily optimized in SciPy
    item_sim = item_vectors_norm.dot(item_vectors_norm.T)

    # --- RAM SAVER: Keep only Top-K neighbors per item ---
    # If you skip this, a large catalog will cause an Out-Of-Memory error.
    print(f"Sparsifying matrix to top-{top_k} neighbors to save RAM...")

    # We only need to do this if the matrix got dense or is overwhelmingly full
    item_sim = item_sim.tolil()
    for i in range(item_sim.shape[0]):
        row = item_sim.data[i]
        if len(row) > top_k:
            # Find the threshold value for the top-Kth element
            threshold_val = np.partition(row, -top_k)[-top_k]
            # Zero out anything below the threshold
            item_sim.data[i] = [val if val >= threshold_val else 0 for val in row]

    # Convert back to CSR for fast math in your main script
    item_sim_sparse = item_sim.tocsr()
    item_sim_sparse.eliminate_zeros()

    print("Item similarity matrix ready!")
    return item_sim_sparse

# Run it:
item_similarity_sparse = fast_item_similarity(train_data_matrix_decay, top_k=100)

Normalizing item vectors...
Computing sparse dot product (this is identical to cosine similarity)...
Sparsifying matrix to top-100 neighbors to save RAM...
Item similarity matrix ready!


In [26]:
# @title Content similarity with sparse dot product

def fast_content_similarity(embeddings, top_k=100):
    print("Normalizing embedding vectors...")
    # Ensure L2 norm is 1
    embeddings_norm = normalize(embeddings, norm='l2', axis=1)

    print("Computing dense dot product...")
    # Pure numpy dot product. Blazing fast.
    content_sim_dense = np.dot(embeddings_norm, embeddings_norm.T)

    print(f"Sparsifying to top-{top_k} neighbors...")
    # Create a mask of the top K values per row
    for i in range(content_sim_dense.shape[0]):
        row = content_sim_dense[i]
        if len(row) > top_k:
            threshold_idx = np.argpartition(row, -top_k)[-top_k]
            threshold_val = row[threshold_idx]
            row[row < threshold_val] = 0

    # Convert to sparse CSR matrix so it plays nicely with your main script
    content_sim_sparse = sparse.csr_matrix(content_sim_dense)
    content_sim_sparse.eliminate_zeros()

    print("Content similarity matrix ready!")
    return content_sim_sparse

# Run it (assuming `my_item_embeddings` is your N x 384 or N x 768 numpy array):
content_similarity = fast_content_similarity(embeddings, top_k=100)

Normalizing embedding vectors...
Computing dense dot product...
Sparsifying to top-100 neighbors...
Content similarity matrix ready!


## A step that isn't shown here:

**Fine-tuning parametrisation using train-test split** We uses the 80/20 split to test our parameters on 'hidden' data (without sorting interactions with time). Computing accuracy with a MAP@10 accuracy helper function.

Parameters we fine-tuned using this method:
* k of knn
* Weight between models
* Min_weight of decay

In [ ]:
# @title Final 3-way hybrid following upgrades, used with any possible weight test (sum to 1)

my_weight_tests = [(0.1, 0.15, 0.75)]

hyper_fast_weight_search(
    train_matrix=train_data_matrix_decay,
    item_sim_matrix=item_similarity_sparse,
    content_sim_matrix=content_similarity,
    weight_combos=my_weight_tests,
    k=20,
    batch_size=250
)

Calculating global item popularity for cold-start users...
Starting hyper-fast grid search for 1 combinations...


# Final words

There is much that can still be done to improve our accuracy !

**How we would go ahead:**
* What about a 4 way hybrid model? (one that actually works as we failed to improve our accuracy with our preliminary attempts).
* Even FINER tuning of parameters (if we want to move the needle by 0.001.)
* Trying different content embedding models.
* Using another API address, one where there is more information on the books! Maybe the same model with books that are more popular (and therefore where we can find their descriptions online) might perform better.